# Dempster-Shafer Evidential Deep Learning (EDL) Multi-Detector Fusion
## Proposal P5 — Cold Chain EWS Digital Twin

**Student:** Krishna Sikheriya (`IIT2023139`)  
**Supervisor:** Netranand Pathak | **Instructor:** Dr. Shashi Kant Rai  
**Repository:** [coldchain-ews-twin](https://github.com/Krishna200608/coldchain-ews-twin)  

### Novelty Claim:
Combines 7 complementary detector signals into Dirichlet belief parameters $(e_\text{normal}, e_\text{alarm}) \ge 0$ via Evidential Deep Learning (Sensoy et al., NeurIPS 2018), computing calibrated alarm probability $\hat{p}$, epistemic vacuity $u = 2/S$, and inter-detector Dempster-Shafer conflict mass $K$.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

### 1. Evidential Deep Learning MLP Architecture and Loss
Maps 7-dimensional multi-detector score vector to non-negative evidence $\mathbf{e} \ge 0$ using Softplus.

In [ ]:
class EvidentialFusionMLP(nn.Module):
    def __init__(self, input_dim=7, hidden_dim=32, latent_dim=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU(),
            nn.Linear(latent_dim, 2),
            nn.Softplus()
        )

    def forward(self, x):
        evidence = self.net(x)
        alpha = evidence + 1.0
        S = alpha.sum(dim=1, keepdim=True)
        prob_alarm = alpha[:, 1:2] / S
        vacuity = 2.0 / S
        return prob_alarm, vacuity, evidence

def evidential_loss(alpha, y_one_hot, lambda_kl=0.01):
    S = alpha.sum(dim=1, keepdim=True)
    ace = torch.sum(y_one_hot * (torch.digamma(S) - torch.digamma(alpha)), dim=1).mean()
    alpha_tilde = y_one_hot + (1.0 - y_one_hot) * alpha
    K = float(alpha.shape[1])
    alpha_0 = alpha_tilde.sum(dim=1, keepdim=True)
    kl = (torch.lgamma(alpha_0) - torch.lgamma(torch.tensor(K, device=alpha.device))
          - torch.lgamma(alpha_tilde).sum(dim=1, keepdim=True)
          + ((alpha_tilde - 1.0) * (torch.digamma(alpha_tilde) - torch.digamma(alpha_0))).sum(dim=1, keepdim=True)).mean()
    return ace + lambda_kl * kl

### 2. Dempster-Shafer Combination Rule
Computes inter-detector conflict mass $K$ when combining IF and LSTM outputs.

In [ ]:
def dempster_shafer_combine(p_if, p_lstm, p_phys):
    m1_a = max(0.0, min(1.0, p_if))
    m1_n = 1.0 - m1_a
    m2_a = max(0.0, min(1.0, p_lstm))
    m2_n = 1.0 - m2_a
    K = m1_a * m2_n + m1_n * m2_a
    denom = max(1e-6, 1.0 - K)
    fused = (m1_a * m2_a) / denom
    normal = (m1_n * m2_n) / denom
    total = fused + normal
    return (fused / total if total > 0 else 0.5), float(K)

### 3. Load Trained Model & Verification
Demonstrates inference on test-split samples with evidential uncertainty quantification.

In [ ]:
import pathlib
REPO_ROOT = pathlib.Path("..")
model = EvidentialFusionMLP().to(device)
ckpt = torch.load(REPO_ROOT / "models/evidential_fusion_out.pt", map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print("Loaded model weights successfully!")

# Forward dummy
x_dummy = torch.randn(5, 7, device=device)
prob, u, evidence = model(x_dummy)
print("Prob alarm:", prob.squeeze().tolist())
print("Vacuity (u):", u.squeeze().tolist())